# Puerta P3 — tamaño de muestra (modelo prenatal)

Etapa del framework: **P3**. Lee `mimiciv.antenatal_predictors_v1` (margen de 7 días) y comprueba con los criterios de Riley et al. si los 129 eventos alcanzan para los **27 parámetros candidatos** de la ficha del desenlace. Versión en inglés: `01_sample_size_gate.ipynb` (mismo código).

**Cómo se decide.** El framework usa el R² de Cox-Snell observado de una regresión logística con validación cruzada: predicciones fuera de la partición, 5 particiones agrupadas por paciente, imputación de faltantes y escalado ajustados solo con el entrenamiento de cada partición. Solo se calcula el R²; el AUROC y los coeficientes no se miran en esta puerta.

**Si no pasa.** Según el framework, primero se ajusta y clasifica el modelo (P4). Si cumple los criterios de P4, se puede reajustar **una vez** con el conjunto reducido de 17 parámetros fijado en la ficha. Si tampoco hay datos para eso, el desenlace se reporta como análisis de factibilidad.

Salida (solo agregada): `data/outputs_01_sample_size_gate.txt`.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # 'penalty' renamed in recent scikit-learn

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.database import get_connect_database, disconnect_database

SEED = 20261119
BUFFER_DAYS = 7          # main information window
N_FOLDS = 5
S_TARGET = 0.9           # Riley criterion 1: expected shrinkage
DELTA = 0.05             # Riley criterion 2: optimism in Nagelkerke R2
MARGIN = 0.05            # Riley criterion 3: precision of the overall risk
OUT_FILE = ROOT / "data" / "outputs_01_sample_size_gate.txt"

## 1. Datos (puerta P1: misma población y eventos)

In [ ]:
engine = get_connect_database()
df = pd.read_sql(
    "SELECT * FROM mimiciv.antenatal_predictors_v1 WHERE buffer_days = %(b)s",
    engine, params={"b": BUFFER_DAYS})
disconnect_database(engine)

assert len(df) == 19440 and df["hadm_id"].is_unique, "population differs from gate G1"
assert int(df["label_main"].sum()) == 129, "events differ from gate G1"
print(len(df), "admissions,", df["subject_id"].nunique(), "patients,", int(df["label_main"].sum()), "events")

## 2. Matriz de diseño: los 27 parámetros (referencias: blanca, privado, casada)

In [ ]:
def design_matrix(d):
    """The 27 candidate parameters of the outcome card (section 4). Reference levels:
    White, Private, Married."""
    X = pd.DataFrame(index=d.index)
    X["age"] = d["age"].astype(float)
    for g in ["Black", "Hispanic", "Asian", "Other or unknown"]:
        X[f"race_{g}"] = (d["race_group"] == g).astype(float)
    X["ins_Medicaid"] = (d["insurance_group"] == "Medicaid").astype(float)
    X["ins_Medicare_other_missing"] = (d["insurance_group"] == "Medicare, other or missing").astype(float)
    X["english"] = d["english"].astype(float)
    X["marital_Not_married"] = (d["marital_group"] == "Not married").astype(float)
    X["marital_Missing"] = (d["marital_group"] == "Missing").astype(float)
    for c in ["any_prior_admission", "prior_admission_90d",
              "hx_cesarean", "hx_hypertensive_pregnancy", "hx_chronic_hypertension",
              "hx_pregestational_diabetes", "hx_gestational_diabetes", "hx_substance_use",
              "hx_thyroid_disease", "hx_adverse_pregnancy_outcome", "hx_other_high_risk_chronic"]:
        X[c] = d[c].astype(float)
    X["last_sbp"] = d["last_sbp"].astype(float)
    X["last_dbp"] = d["last_dbp"].astype(float)
    X["any_elevated_bp"] = d["any_elevated_bp"].astype(float)
    X["last_bmi"] = d["last_bmi"].astype(float)
    X["last_hemoglobin"] = d["last_hemoglobin"].astype(float)
    X["had_prenatal_care_here"] = d["had_prenatal_care_here"].astype(float)
    return X

X = design_matrix(df)
y = df["label_main"].astype(int).to_numpy()
groups = df["subject_id"].to_numpy()
P = X.shape[1]
assert P == 27, P
print("candidate parameters:", P)
print("missing values:", X.isna().sum()[X.isna().sum() > 0].to_dict())

## 3. Regresión logística con validación cruzada agrupada por paciente

In [ ]:
# Out-of-fold predictions of an unpenalized logistic regression. Imputation and scaling are
# fitted on the training part of each split only; folds are grouped by patient.
def make_model():
    return make_pipeline(
        IterativeImputer(max_iter=10, random_state=SEED),
        StandardScaler(),
        LogisticRegression(penalty=None, max_iter=5000))

oof = np.full(len(y), np.nan)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (tr, te) in enumerate(cv.split(X, y, groups), 1):
    m = make_model().fit(X.iloc[tr], y[tr])
    oof[te] = m.predict_proba(X.iloc[te])[:, 1]
    print(f"fold {k}: {len(te)} admissions, {y[te].sum()} events")
assert not np.isnan(oof).any()

## 4. R² de Cox-Snell observado

In [ ]:
def loglik(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

n, E = len(y), int(y.sum())
phi = E / n
L0 = n * (phi * np.log(phi) + (1 - phi) * np.log(1 - phi))   # intercept-only model
L1 = loglik(y, oof)
R2_cs = 1 - np.exp(2 * (L0 - L1) / n)
R2_max = 1 - np.exp(2 * L0 / n)
print(f"prevalence {phi:.4f}; max Cox-Snell R2 {R2_max:.4f}")
print(f"observed (cross-validated) Cox-Snell R2 {R2_cs:.5f} = {100 * R2_cs / R2_max:.1f}% of the maximum")

## 5. Criterios de Riley et al.

1. Encogimiento esperado de 0,9 o más.
2. Diferencia de 0,05 o menos entre el R² de Nagelkerke aparente y el ajustado.
3. Riesgo global estimado con un margen de ±0,05.

`p_supported` es el número de parámetros que soportan los datos con el R² observado.

In [ ]:
# Riley et al. (BMJ 2020) criteria for a binary outcome
def riley(p, R2, R2max, phi):
    if R2 <= 0:
        return None
    n1 = p / ((S_TARGET - 1) * np.log(1 - R2 / S_TARGET))
    S2 = R2 / (R2 + DELTA * R2max)
    n2 = p / ((S2 - 1) * np.log(1 - R2 / S2))
    n3 = (1.96 / MARGIN) ** 2 * phi * (1 - phi)
    S = max(S_TARGET, S2)
    p_max = int(np.floor(n * (S - 1) * np.log(1 - R2 / S)))   # parameters the data support
    n_req = int(np.ceil(max(n1, n2, n3)))
    return dict(n1=n1, n2=n2, n3=n3, S2=S2, n_required=n_req,
                events_required=int(np.ceil(n_req * phi)), p_supported=p_max)

res = riley(P, R2_cs, R2_max, phi)
if res is None:
    gate = "NOT PASSED (cross-validated R2 is not positive)"
else:
    gate = "PASSED" if n >= res["n_required"] else "NOT PASSED"
    for k, v in res.items():
        print(f"{k}: {v:.3f}" if isinstance(v, float) else f"{k}: {v}")
print(f"available: {n} admissions, {E} events")
print("gate G3 with", P, "parameters:", gate)

## 6. Salida agregada

In [ ]:
lines = [
    "Gate G3 (sample size), antenatal model; aggregate output only",
    f"buffer_days={BUFFER_DAYS}; folds={N_FOLDS} grouped by patient; seed={SEED}",
    f"admissions={n}; events={E}; prevalence={phi:.4f}; candidate_parameters={P}",
    f"R2_cs_max={R2_max:.4f}; R2_cs_observed_cv={R2_cs:.5f} ({100 * R2_cs / R2_max:.1f}% of max)",
]
if res is not None:
    lines += [f"n_required={res['n_required']} (criterion1={res['n1']:.0f}, criterion2={res['n2']:.0f}, criterion3={res['n3']:.0f}); "
              f"events_required={res['events_required']}; parameters_supported={res['p_supported']}"]
lines += [f"gate={gate}"]
OUT_FILE.parent.mkdir(exist_ok=True)
OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))